In [5]:
# Scenario: Customer Support Chatbot Workflow
# Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

# - State Definition (BotState)
# - The chatbot keeps track of:
# - The question asked by the customer.
# - The answer generated.
# - The history of all past questions.
# - Think of this as the chatbot’s notebook where it records the conversation.

# - Nodes (Functions)
# - get_answer:
# When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
# It also adds the question to the history log.
# - format_output:
# Before sending the response back to the customer, the chatbot reformats it into a friendly style:
# “Bot says: Answer to: What are your store hours?”

# - Graph Flow
# - The chatbot starts at the get_answer node (entry point).
# - Once the answer is generated, it flows to the format_output node.
# - Finally, the conversation ends at END, meaning the chatbot has
#  delivered its response.


from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

In [13]:
# Scenario: Customer Support Chatbot (Question-Based)
# Imagine a company has deployed a chatbot that answers customer
#  questions by calling the Groq API. The workflow is modeled as a
#  graph of states, where each customer query flows through nodes until
#   a response is delivered.

# 1. State Definition
# The chatbot maintains a notebook-like state:
# - question → The customer’s query.
# - answer → The response generated by Groq.
# - history → A log of all past questions.

from langgraph.graph import StateGraph, END
from typing import TypedDict
from openai import OpenAI
from google.colab import userdata

# ✅ Get OpenAI API Key
api_key = userdata.get('OPENAI_API_KEY')

# ✅ Initialize OpenAI client
client = OpenAI(api_key=api_key)

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes
def get_answer(state: BotState):
    q = state["question"]

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful customer support assistant."},
                {"role": "user", "content": q}
            ]
        )

        ans = response.choices[0].message.content

    except Exception as e:
        ans = f"Error: {str(e)}"

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }


def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}


# 3. Build Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Flow
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


# 5. Run
if __name__ == "__main__":
    app = graph.compile()

    state = {
        "question": "What are your store hours?",
        "answer": "",
        "history": []
    }

    result = app.invoke(state)

    print(result["answer"])

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [9]:
# Scenario: Customer Support Call Center
# A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

# 1. State Definition (SupportState)
# The chatbot keeps track of:
# - query → What the customer asked.
# - category → Which department it belongs to (billing, tech, general).
# - response → What the bot replies with.
# Think of this as the customer’s “ticket form.”

# 2. Router Node (route_query)
# When a customer types a question, the router scans it:
# - If the query mentions “bill” or “payment”, it routes to billing_agent.
# - If it mentions “error” or “bug”, it routes to tech_agent.
# - Otherwise, it defaults to general_agent.
# This is like a receptionist deciding which desk you should go to.

# 3. Agent Nodes
# - billing_agent → Replies with “Billing dept: [query]”.
# - tech_agent → Replies with “Tech support: [query]”.
# - general_agent → Replies with “General help: [query]”.
# Each agent specializes in its own type of problem.

# 4. Graph Flow
# - Entry point: router.
# - Router decides the path based on the query.
# - The query flows into the correct agent node.
# - The agent generates a response and ends the conversation.


from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str   # "billing" | "tech" | "general"
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}

# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",    # from node
    route_query, # function that returns next node
    {            # mapping: return value → node name
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)


In [11]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import google.generativeai as genai
from google.colab import userdata

# Configure Gemini
api = userdata.get('faltu_kaam1')
genai.configure(api_key=api)

# 1. Define State
class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# 2. Helper: Gemini Call
def gemini_call(prompt: str):
    try:
        model = genai.GenerativeModel("gemini-1.5-flash")
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error: {str(e)}"

# 3. Nodes
def search_web(state: ResearchState):
    print(f"🔍 Searching: {state['topic']}")

    new_results = [
        gemini_call(f"Give me a short article snippet about {state['topic']}"),
        gemini_call(f"Give me another snippet about {state['topic']}")
    ]

    results = state["search_results"] + new_results

    return {
        "search_results": results,
        "steps_done": state["steps_done"] + 1
    }

def analyze_results(state: ResearchState):
    print(f"📊 Analyzing {len(state['search_results'])} results")

    joined_results = "\n".join(state["search_results"])

    analysis = gemini_call(
        f"Analyze these sources and extract key insights:\n{joined_results}"
    )

    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }

def summarize(state: ResearchState):
    print("✍️ Generating summary...")

    summary = gemini_call(
        f"Summarize this analysis in simple terms:\n{state['analysis']}"
    )

    return {"summary": summary}

def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"
    return "summarize"

# 4. Build Graph
g = StateGraph(ResearchState)

g.add_node("search_web", search_web)
g.add_node("analyze", analyze_results)
g.add_node("summarize", summarize)

g.set_entry_point("search_web")

g.add_edge("search_web", "analyze")

g.add_conditional_edges(
    "analyze",
    should_continue,
    {
        "search_web": "search_web",
        "summarize": "summarize"
    }
)

g.add_edge("summarize", END)

# 5. Run
if __name__ == "__main__":
    app = g.compile()

    result = app.invoke({
        "topic": "Quantum Computing",
        "search_results": [],
        "analysis": "",
        "summary": "",
        "steps_done": 0
    })

    print("\n✅ Final Summary:\n", result["summary"])

🔍 Searching: Quantum Computing


📊 Analyzing 2 results


🔍 Searching: Quantum Computing


📊 Analyzing 4 results


✍️ Generating summary...



✅ Final Summary:
 Error: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.


In [1]:
# Scenario: AI-Assisted Email Workflow (Question-Based)
# Context
# A company deploys an AI-powered email assistant to help employees draft, review, and send professional emails.
# The workflow is modeled as a graph of states, where each email task flows through nodes until it is either approved
# and sent or rejected.

# 1. State Definition
# The assistant maintains a notebook-like state:
# - task → The subject or purpose of the email (e.g., "Q3 Report").
# - draft → The AI-generated email draft.
# - approved → A flag indicating whether the human reviewer has approved the draft.

# 2. Workflow (Graph of States)
# Each email task flows through nodes:
# - Draft Node
# - AI generates a draft email based on the task.
# - Updates draft.
# - Review Node (Interrupt)
# - Execution pauses here.
# - Human reviewer inspects the draft and decides whether to approve or reject.
# - Updates approved.
# - Send Node
# - If approved = True → Email is sent.
# - If approved = False → Email is rejected.
# - Updates task with final status (SENT or REJECTED).
# - End Node
# - Workflow completes.

# 3. Example Flow
# - Employee: "Draft an email for the Q3 Report."
# - State: task = "Q3 Report"
# - Assistant drafts:
# Dear User,
# Regarding: Q3 Report
# [AI drafted content]
# - Human reviews → Approves draft (approved = True)
# - Assistant sends → task = "SENT: Q3 Report"
# - Final Output: ✅ Email delivered.

# 👉 Scenario Question:
# "Imagine you are designing an AI-powered email assistant that drafts emails, pauses for human review, and
# then either sends or rejects them. How would you structure the state and workflow graph to ensure accountability
#  and human oversight in the process?"

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import google.generativeai as genai
from google.colab import userdata

# 🔑 Configure Gemini
api = userdata.get("faltu_kaam1")
genai.configure(api_key=api)

# 1. Define State
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool

# 2. Gemini Call
def gemini_call(prompt: str):
    try:
        model = genai.GenerativeModel("gemini-1.5-flash")
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error: {str(e)}"

# 3. Nodes

# 📝 Draft Node
def draft_email(state: EmailState):
    print(f"📝 Drafting email for task: {state['task']}")

    draft = gemini_call(
        f"Write a professional email regarding: {state['task']}"
    )

    return {"draft": draft}


# 👨‍💼 Human Review Node (HITL)
def human_review(state: EmailState):
    print(f"\n📧 Draft ready for review:\n\n{state['draft']}\n")
    return {}  # Pause happens here


# 📤 Send Node
def send_email(state: EmailState):
    if state.get("approved", False):
        print("✅ Email approved and sent.")
        return {"task": f"SENT: {state['task']}"}
    else:
        print("❌ Email rejected.")
        return {"task": f"REJECTED: {state['task']}"}


# 4. Build Graph
g = StateGraph(EmailState)

g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("send", send_email)

g.set_entry_point("draft")

g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)


# 5. Enable Memory + Interrupt
checkpointer = MemorySaver()

app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]  # ⏸️ pause before human review
)


# 6. Run Workflow

thread = {"configurable": {"thread_id": "email-1"}}

# 🚀 Step 1: AI drafts email
app.invoke({
    "task": "Q3 Report Update",
    "draft": "",
    "approved": False
}, thread)

# 🧑 Step 2: Human approves/rejects
app.invoke({
    "approved": True   # change to False to test rejection
}, thread)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


📝 Drafting email for task: Q3 Report Update


📝 Drafting email for task: Q3 Report Update


{'task': 'Q3 Report Update',
 'draft': 'Error: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.',
 'approved': True}

In [2]:
"""Scenario Question
"Imagine you are designing an AI-powered assistant that drafts structured reports, pauses for human review, and then either publishes or rejects them.
How would you structure the state and workflow graph to ensure accountability and human oversight in the process?"
"""
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import google.generativeai as genai
from google.colab import userdata

# 🔑 Configure Gemini API
api = userdata.get("faltu_kaam1")
genai.configure(api_key=api)

# ================================
# 1. Define State
# ================================
class ReportState(TypedDict):
    task: str
    draft: str
    approved: bool
    review_comments: str
    status: str

# ================================
# 2. Gemini Call
# ================================
def gemini_call(prompt: str):
    try:
        model = genai.GenerativeModel("gemini-1.5-flash")
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error: {str(e)}"

# ================================
# 3. Nodes
# ================================

# 📝 Draft Node
def draft_report(state: ReportState):
    print(f"📝 Generating report for: {state['task']}")

    draft = gemini_call(
        f"Write a structured professional report on: {state['task']}"
    )

    return {
        "draft": draft,
        "status": "DRAFT_CREATED"
    }


# 👨‍💼 Human Review Node (HITL)
def review_report(state: ReportState):
    print("\n📄 Draft Report:\n")
    print(state["draft"])
    print("\n--- Awaiting Human Review ---\n")

    return {}  # pause here


# ⚖️ Decision Node
def decision_node(state: ReportState):
    if state.get("approved", False):
        return {"status": "APPROVED"}
    else:
        return {"status": "REJECTED"}


# 📤 Publish Node
def publish_report(state: ReportState):
    print("✅ Report Approved and Published")
    return {
        "status": f"PUBLISHED: {state['task']}"
    }


# ❌ Reject Node
def reject_report(state: ReportState):
    print("❌ Report Rejected")
    return {
        "status": f"REJECTED: {state['task']}"
    }

# ================================
# 4. Build Graph
# ================================
g = StateGraph(ReportState)

g.add_node("draft", draft_report)
g.add_node("review", review_report)
g.add_node("decision", decision_node)
g.add_node("publish", publish_report)
g.add_node("reject", reject_report)

# Flow
g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "decision")

# Conditional Routing
g.add_conditional_edges(
    "decision",
    lambda state: "publish" if state.get("approved") else "reject",
    {
        "publish": "publish",
        "reject": "reject"
    }
)

g.add_edge("publish", END)
g.add_edge("reject", END)

# ================================
# 5. Enable Memory + Interrupt
# ================================
checkpointer = MemorySaver()

app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]   # ⏸️ pause before review
)

# ================================
# 6. Run Workflow
# ================================
thread = {"configurable": {"thread_id": "report-1"}}

# 🚀 Step 1: Generate Draft
app.invoke({
    "task": "Impact of AI on Healthcare",
    "draft": "",
    "approved": False,
    "review_comments": "",
    "status": ""
}, thread)

# 🧑 Step 2: Human Review (resume)
app.invoke({
    "approved": True,   # change to False to test rejection
    "review_comments": "Looks good"
}, thread)

📝 Generating report for: Impact of AI on Healthcare


📝 Generating report for: Impact of AI on Healthcare


{'task': 'Impact of AI on Healthcare',
 'draft': 'Error: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.',
 'approved': True,
 'review_comments': 'Looks good',
 'status': 'DRAFT_CREATED'}